In [ ]:
import napari
from skimage.registration import phase_cross_correlation
import dask.array as da
import numpy as np

def center_pad_nd(fov, target_shape):
    pad_widths = []
    for fov_dim, target_dim in zip(fov.shape, target_shape):
        total_pad = target_dim - fov_dim
        pad_before = total_pad // 2
        pad_widths.append((pad_before, total_pad - pad_before))
    
    return np.pad(fov, pad_widths, mode='constant')

def place_fov(fov, target_shape, shift):
    new = np.zeros(target_shape, dtype=fov.dtype)
    
    start_y = int(((target_shape[0] - fov.shape[0]) // 2) + shift[0])
    start_x = int(((target_shape[1] - fov.shape[1]) // 2) + shift[1])
    start_z = int(((target_shape[2] - fov.shape[2]) // 2) + shift[2])
    
    new[start_y:start_y + fov.shape[0], start_x:start_x + fov.shape[1],start_z:start_z + fov.shape[2]] = fov
    return new


full = da.from_zarr(r"X:\Microglia\E4FAD2\2025_12_01_E4FAD2_Iba635_640_nkt1line_700_MFS4_4x4_10ms_out\fused\2025_12_01_E4FAD2_Iba635_640_nkt1line_700_MFS4_4x4_10ms_fused.zarr\0")[0,0,...]
full_ds = da.coarsen(np.mean, full, {0: 2,1: 2, 2: 2}, trim_excess = True)


dws = [r"X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0",
       r"X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_1_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0",
       r"X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_2_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0"]


placed_fovs = []

for dw_path in dws:
    print(dw_path)
    dw0 = da.from_zarr(dw_path)
    dw_downsampled = da.coarsen(np.mean, dw0, {0: 16,1: 16, 2: 16}, trim_excess = True)
    padded_fov = center_pad_nd(dw_downsampled,full_ds.shape)
    shift, _, _ = phase_cross_correlation(full_ds, padded_fov, upsample_factor=4)
    print(shift)
    placed_fov = place_fov(dw_downsampled,full_ds.shape,shift) # or try with the translate parameter in add image

    placed_fovs.append(placed_fov)

viewer = napari.Viewer()
viewer.add_image(full_ds)

for placed_fov in placed_fovs:
    viewer.add_image(placed_fov)


#this will be done with cell coordinates in the future once the shift and scaling is found, and with the fused deconvolved volumes

X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0
[  77.5 -179.   -32.5]
X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_1_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0
[306.75 -61.25 -13.25]
X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_2_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0
[189.25 184.5   12.25]


Rendering frames...


100%|██████████| 501/501 [05:12<00:00,  1.60it/s]
